# Projet 5MA-OSI : Transfert d'organes en domino sous incertitudes

## Introduction

Malgré l'augmentation croissante du nombre de transplantations d'organes effectuées chaque année (environ 6000 en 2017 dont 3782 transplantations de reins), la demande reste en perpétuelle augmentation. Ainsi 6000 organes, dont 3782 reins, ont été transplantés en 2017, mais il y avait encore 24000 personnes en attente d'un organe la même année. Les organes transplantés peuvent provenir d'un donneur décédé ou, dans le cas des reins et du foie, d'un donneur vivant consentant, le plus souvent membre de la famille du patient. Hélas, même si un proche accepte de prendre ce risque pour sa santé, il ne sera pas forcément compatible avec le patient. Pour cette raison, les pratiques médicales et les législations évoluent dans de nombreux pays afin de permettre la mise en place d'un programme d'échange de dons d'organes.

L'exemple le plus simple d'échange de don d'organes est celui où deux patients $P_1$ et $P_2$ sont accompagnés de donneurs $D_1$ et $D_2$. Les patients sont supposés incompatibles avec les donneurs qui les accompagnent, mais on suppose que $D_1$ est compatible avec $P_2$ et $D_2$ avec $P_1$. Il est alors possible de transplanter un organe de $D_1$ vers $P_2$ et de $D_2$ vers $P_1$ avec le consentement de tous et en suivant la procédure légale.

Plus généralement, un cycle d'échange d'organes associe $k$ paires de patient-donneur $(P_{i_1},D_{i_1}), \dots, (P_{i_k},D_{i_k})$ de sorte que $D_{i_l}$ donne à $P_{i_{l+1}}$ pour $l=1,\dots,k-1$ et $D_{i_k}$ donne à $P_{i_1}$.
Par ailleurs, le point essentiel est que les transferts soient tous réalisés en même temps et dans le même hôpital pour éviter qu'une rétractation de dernière minute ne lèse un patient et son donneur, et que les patients et donneurs venus ensemble et leur famille puissent se soutenir émotionnellement durant l'hospitalisation. 
Pour cette raison, le nombre d'échanges prenant place au sein d'un même cycle est nécessairement limité. En pratique, l'organisation d'un cycle de trois paires est déjà une épreuve pour le personnel d'un hôpital, et le plus grand cycle ayant jamais eu lieu a a impliqué six patients et donneurs.

Dans ce projet, nous prendrons le point de vue de l'organisme national responsable de la gestion du programme d'échange d'organes. 
À chaque phase d'échange, l'objectif de cet organisme est de choisir un ensemble de cycles d'échanges entre paires compatibles afin de maximiser le nombre de patients recevant un organe. Dans certains cas, on peut aussi donner une priorité à certains patients en fonction de la gravité de leur état ou de la durée de leur attente. 
Pour cela, on pourra attribuer des poids différents à chaque patient et maximiser la somme des poids des patients recevant un organe. 

## Étude du problème déterministe


### Jeux de données

Tous vos tests seront basés sur les jeux de données de la [PrefLib](https://www.preflib.org/dataset/00036). Ces jeux de données ne correspondent pas à des programmes d'échanges d'organes réels pour des raisons de confidentialité, mais ils reproduisent la structure des données réelles plus fidèlement que les données aléatoires utilisées pour le projet de RO. Ils 
sont constituées d'informations individuelles et d'un graphe de compatibilité. Chaque fichier .wmd décrit un graphe de compatibilité orienté, $G=(V,A)$, où chaque sommet de $V$ représente une paire patient-donneur et où un arc entre deux paires $(P_k,D_k)$ et $(P_l,D_l)$ signifie que $D_k$ est compatible avec $P_l$. La compatibilité est obtenue à partir des données biologiques individuelles (e.g., les groupes sanguins) et d'un _test croisé_ lors duquel des biologistes mettent en présence des tissus d'un malade et d'un donneur supposé. Chaque fichier est composé comme suit :  
- les 11 premières ligne contiennent diverses informations sur le fichier, dont le nombre de sommets ($n$) et le nombre d'arcs ($m$),
- les $n$ lignes suivantes nomment les sommets (un sommet par paire),
- les $m$ lignes restantes contiennent les arcs du graphe et le poids de chaque arc. Le poids de chaque arc indique l'urgence de la situation du malade de la paire destination. 

Je fournis ci-dessous une fonction permettant de lire les fichiers .wmd et retournant un graphe de compatibilité pondéré par les poids de chaque patient. Pour vous aider à vous lancer, je fournis déjà quelques instances en accompagnement de ce sujet sur Moodle. Elles contiennent un nombre croissant de paires patient-donneur allant de 16 à 512.

In [1]:
# Add all the necessary packages. This can be commented out or even deleted once done on your computer
import Pkg; 
Pkg.add(["Graphs", "JuMP","HiGHS","DelimitedFiles","Distributions","NBInclude"]);

   Resolving package versions...
   Installed Rmath_jll ─────────────── v0.5.2+0
   Installed CompositionsBase ──────── v0.1.2
   Installed NBInclude ─────────────── v2.4.0
   Installed CommonSolve ───────────── v0.2.14
   Installed MacroTools ────────────── v0.5.16
   Installed DiffResults ───────────── v1.1.0
   Installed ForwardDiff ───────────── v1.5.0
   Installed DataStructures ────────── v0.19.7
   Installed Bzip2_jll ─────────────── v1.0.9+0
   Installed DelimitedFiles ────────── v1.9.1
   Installed FillArrays ────────────── v1.19.0
   Installed HiGHS ─────────────────── v1.27.0
   Installed ConstructionBase ──────── v1.6.1
   Installed HypergeometricFunctions ─ v0.3.31
   Installed AliasTables ───────────── v1.1.3
   Installed ArnoldiMethod ─────────── v0.4.0
   Installed CodecZlib ─────────────── v0.7.9
   Installed StatsBase ─────────────── v0.34.14
   Installed DataAPI ───────────────── v1.16.0
   Installed OrderedCollections ────── v2.0.3
   Installed Statistics ──────────

In [2]:
# Include the necessary packages
using Random, Graphs, JuMP, HiGHS, DelimitedFiles, Distributions

In [3]:
# Exemple d'inclusion d'un notebook : tout le notebook est exécuté
# using NBInclude
# @nbinclude("KEP_readfile.ipynb")

# Exemple en incluant directement un fichier Julia : ici, j'ai ajouté les lignes de test dans le fichier .jl afin d'illustrer que les sorties du fichier sont redirigés vers ce notebook au moment où il est inclus
include("KEP_readfile.jl")

read_wmd_file (generic function with 1 method)

In [4]:
# Exemple d'utilisation de la fonction supposant que les fichiers sont dans le dossier data_KEP
G, edge_weight = read_wmd_file("data_KEP/KEP_001.wmd");
println("Number of vertices: ", nv(G));
println("Number of arcs: ", ne(G));
println("List of arcs with weights:")
for e in edges(G)
    print("($(e.src),$(e.dst)):$(edge_weight[e.src,e.dst]), ");
end

Number of vertices: 16
Number of arcs: 59
List of arcs with weights:
(1,5):1.0, (1,6):1.0, (2,1):1.0, (2,3):1.0, (2,11):1.0, (2,12):1.0, (3,5):1.0, (3,8):1.0, (4,1):1.0, (4,3):1.0, (4,11):1.0, (6,1):1.0, (6,3):1.0, (6,11):1.0, (7,1):1.0, (7,3):1.0, (7,11):1.0, (7,12):1.0, (8,1):1.0, (8,3):1.0, (8,11):1.0, (9,1):1.0, (9,3):1.0, (9,11):1.0, (9,12):1.0, (10,1):1.0, (10,2):1.0, (10,3):1.0, (10,5):1.0, (10,8):1.0, (10,9):1.0, (10,11):1.0, (10,12):1.0, (10,14):1.0, (10,15):1.0, (10,16):1.0, (12,1):1.0, (12,3):1.0, (12,11):1.0, (13,2):1.0, (13,3):1.0, (13,5):1.0, (13,7):1.0, (13,8):1.0, (13,9):1.0, (13,11):1.0, (13,14):1.0, (13,15):1.0, (13,16):1.0, (14,1):1.0, (14,3):1.0, (14,11):1.0, (14,12):1.0, (15,1):1.0, (15,3):1.0, (15,11):1.0, (16,5):1.0, (16,6):1.0, (16,8):1.0, 

### Formulations compactes

Dans le cadre du TD/TP du cours de Recherche Opérationnelle, il vous a été demandé de coder trois formulations PLNE, à savoir :
- celle sans contrainte sur la longueur des cycles
- celle avec des cycles de taille 2
- celle avec la décomposition par hôpital pour des cycles de taille 2 à 6.

J'ai recopié la correction distribuée sur Moodle dans le fichier formulation_compactes.jl. Je l'inclus ci dessous. Afin de prendre le sujet en main, comparer les solutions des 3 modèles sur un ensemble de jeux de données bien choisis. Analyser les résultats d'un point de vue informatique, pratique et sociétal.

In [5]:
# inclusion des fonctions de résolution des formulations compactes
include("formulations_compactes.jl")

# tests de bon fonctionnement sur le petit graphe importé plus haut
solve_cycle_infini(G, edge_weight);
solve_cycle_2(G, edge_weight);
solve_cycle_K(G, edge_weight, 4);


-------------------------------
Résolution du modèle sans contrainte de longueur de cycle

Statut de la résolution : OPTIMAL
Durée de la résolution : 0.3533162
Nombre de transferts réalisés : 4.0
Liste des transferts réalisés :
1 -> 6 ; 3 -> 8 ; 6 -> 1 ; 8 -> 3 ; 
-------------------------------


-------------------------------
Résolution du modèle avec cycles de taille 2

Statut de la résolution : OPTIMAL
Durée de la résolution : 0.0004668
Nombre de transferts réalisés : 4.0
Liste des transferts réalisés :
1 <-> 6 ; 3 <-> 8 ; 
-------------------------------


-------------------------------
Résolution du modèle avec cycle de taille au plus 4

Statut de la résolution : OPTIMAL
Durée de la résolution : 0.0040184
Nombre de transferts réalisés : 4
Liste des transferts réalisés :
1 -> 6 ; 3 -> 8 ; 6 -> 1 ; 8 -> 3 ; 
-------------------------------



### Génération de colonnes

L'objectif de cette partie du projet est de coder une méthode de génération de colonnes pour le problème de dons d'organes en dominos (KEP pour Kidney Exchange Problem en anglais) sans incertitudes. Votre code s'appuiera sur les éléments vus en CM et leur application au KEP vue en TD. Pour compléter cette présentation, [une page de la documentation de JuMP](https://jump.dev/JuMP.jl/stable/tutorials/algorithms/cutting_stock_column_generation/) est dédiée à la génération de colonnes. Elle vous offre une autre entrée en matière sur la question.

1. Commencer par écrire le _modèle par cycles_ qui, pour un ensemble de cycles $C$, définit une variable binaire $x_c$ pour chaque cycle $c \in C$, tel que $x_c=1$ si et seulement si le cycle de dons d'organes $c$ est réalisé. Coder une fonction qui, pour un entier $K$ donnée, énumère tous les cycles de taille $K$ ou moins. Et coder une seconde fonction qui résout le modèle par cycle pour un ensemble de cycles $C$ donné. Tester ces fonctions sur des jeux de données que vous choisirez pour $K=2,3,4$.
2. Coder une méthode de génération de colonnes pour résoudre la relaxation linéaire de la formulation par cycles. Vous implémenterez deux versions de la fonction de _pricing_ (résolution du sous-problème) : une ou le sous-problème est formulé à l'aide d'un ensemble de PLNE, et une ou le sous-problème est résolu par un algorithme de plus long chemin.
3. Coder une fonction qui résout le problème de dons en dominos de façon approchée à l'aide de l'heuristique de génération de colonnes décrite en CM.


In [ ]:
cycles = simplecycles_limited_length(G, 4);
y = zeros(Int64, nv(G), length(cycles))
for i in 1:length(cycles)
    for v in cycles[i]
        y[v,i] = 1
    end
end
println("Cycles found: ", cycles)
println("y matrix: ", y)
println(y[5,:])




Cycles found: [[1, 6], [1, 6, 3, 8], [3, 8]]
y matrix: [1 1 0; 0 0 0; 0 1 1; 0 0 0; 0 0 0; 1 1 0; 0 0 0; 0 1 1; 0 0 0; 0 0 0; 0 0 0; 0 0 0; 0 0 0; 0 0 0; 0 0 0; 0 0 0]
16
[0, 0, 0]


In [23]:
function solve_given_cycles(cycles, y) 
 
    I =  1:size(y, 1)
    C = 1:length(cycles)

    model = Model(HiGHS.Optimizer);

    @variable(model, x[c in C], Bin)

    @objective(model, Max, sum(x[c] * y[i,c] for i in I, c in C))

    @constraint(model, [i in I], sum(y[i,c]*x[c] for c in C) <= 1)

    set_silent(model); # je désactive le slog de HiGHS pour que les sorties soient courtes, mais il peut être intéressant de les afficher pour essayer de comprendre ce que le solveur dit, le cours doit vous permettre de comprendre au moins une partie des messages
    timer = @timed optimize!(model);

    # Affichage des résultats
    println("\n-------------------------------")
    println("Résolution du modèle sans contrainte de longueur de cycle\n")
    println("Statut de la résolution : ", termination_status(model));
    println("Durée de la résolution : $(timer.time)")
    if (termination_status(model) == MOI.OPTIMAL)
        println("Nombre de transferts réalisés : ", objective_value(model));
        println("Liste des transferts réalisés :")
        println("Valeur du vecteur x :", value(x))
        for c in C
            if value(x[c]) > 0.5
                println("Cycle ", c, ": ", cycles[c])
            end
        end

    end
    println("\n-------------------------------\n")
end

solve_given_cycles (generic function with 1 method)

In [24]:
solve_given_cycles(cycles, y)


-------------------------------
Résolution du modèle sans contrainte de longueur de cycle

Statut de la résolution : OPTIMAL
Durée de la résolution : 0.0005681
Nombre de transferts réalisés : 4.0
Liste des transferts réalisés :
Valeur du vecteur x :1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, 1:3
And data, a 3-element Vector{Float64}:
  1.0
 -0.0
  1.0
Cycle 1: [1, 6]
Cycle 3: [3, 8]

-------------------------------



### Expériences numériques

#### Protocole de test

Décrire un  protocole de test permettant l'évaluation du travail fait plus haut. 

1. Dites d'abord à quelles questions vous souhaitez répondre.
2. Décrivez ensuite votre benchmark en justifiant sa pertinence.
3. Annoncez les tests que vous comptez faire et expliquez comment ces tests peuvent répondre à vos questions.

#### Résultats et discussion

Une fois le protocole défini et les tests réalisés vient la phase qui démontre l'utilité de tout le travail fourni jusqu'ici, celle de l'exploitation de vos résultats. En premier lieu, il faut parvenir à présenter de façon compacte, claire et complète les résultats qui vous permettront de tirer des conclusions utiles. Beaucoup de choses peuvent sembler de l'ordre du détail à cette étape, mais elles font vite la différence entre des résultats lisibles et d'autres incompréhensibles. Je rappelle par exemple que :
- le choix des couleurs importe sur les graphes
- tous les graphes sont accompagnés de titre, des informations utiles sur les axes des abscisses et ordonnées, et de ce à quoi correspondent les différentes courbes s'il y en a plusieurs
- l'échelle choisie fera plus ou moins bien apparaître les conclusions que l'on souhaite tirer (parfois les mauvais résultats d'une méthode écraseront totalement les différences entre d'autres méthodes testées)
- tout tableau, figure ou graphe apparaissant dans un document doit être clairement référencée dans le corps du texte (c'est aussi vrai pour les références bibliographiques)
- lors du CM sur les protocoles de tests, je vous présenterai les profil de performance ; ils s'avèrent souvent utiles lorsque l'on compare plus de deux méthodes différentes, que les méthodes ne résolvent pas toutes les instances dans la limite de temps ou que de grandes variations des temps de calcul apparaissent. Un package Julia permet de les tracer : [BenchmarkProfiles](https://jso.dev/BenchmarkProfiles.jl/stable/).
- parfois un tableau ou une figure n'apporte pas grand chose et il peut suffire d'indiquer une moyenne ou un autre agrégat important dans le texte 

Finalement, chaque tableau/graphe/figure doit être décrit et commenté dans votre rapport. L'objectif est de répondre de façon argumentée aux questions essentielles que vous aurez sélectionné au moment de la définition du protocole. Si au final les résultats ne sont pas suffisants pour répondre à ces questions, il faut pouvoir identifier les tests qui auraient pu être utiles, puis soit faire ces nouveaux tests, soit les laisser en perspective en fonction de l'ensemble des questions auxquelles vous avez déjà répondu, de votre intérêt pour le problème et du temps dont vous disposez. Si aucune des méthodes n'est totalement satisfaisante du point de vue du problème traité, indiquez, si vous le pouvez, des méthodes ou des améliorations qui pourraient être explorées dans le futur. C'est aussi là que vous aurez l'occasion de discuter d'éventuelles précautions à prendre pour l'utilisation en contexte réel d'une telle méthode automatique d'aide à la décision. 

__Si toutes les tâches précédentes sont nécessaires pour en arriver là, il s'agit bien d'une étape essentielle d'un projet de mathématiques appliquées. C'est en la lisant que l'on juge de l'utilité du travail réalisé !__

1. Présentez les résultats de vos tests sous une forme appropriée.
2. Discutez vos résultats et indiquer vos conclusions pour l'ensemble de l'étude du problème déterministe de dons d'organes en dominos.